In [26]:
import pandas as pd
import numpy as np
from preprocessor import Preprocessor
from sklearn.model_selection import train_test_split

df_neg = pd.read_csv('../data/processedNegative.csv')
df_ntr = pd.read_csv('../data/processedNeutral.csv')
df_pos = pd.read_csv('../data/processedPositive.csv')

raw_tweets = (
    df_neg.columns.to_list()
    + df_ntr.columns.to_list()
    + df_pos.columns.to_list()
)

y_ = [
    (df_neg.columns.size, 0),
    (df_ntr.columns.size, 1),
    (df_pos.columns.size, 2)
]

y = []
for count, label in y_:
    y.extend([label] * count)

y = np.array(y)

proc_binary = Preprocessor(vectorization='binary')
proc_bow    = Preprocessor(vectorization='bow')
proc_tf_idf = Preprocessor(vectorization='tf-idf')

# Show examples
print("Raw:", raw_tweets[:3], '\n')

cleaned   = proc_binary.clean(raw_tweets)
tokenized  = proc_binary.tokenize(cleaned)

lemmatized  = proc_binary.lemmatize(tokenized)
stemmed     = proc_binary.stemming(tokenized)
stemmed_plus = proc_binary.stemming_plus(tokenized)


print("Lemmatized:", lemmatized[:3], '\n')
print("Stemmed: ", stemmed[:3], '\n')
print("Stem+: ", stemmed_plus[:3], '\n')

lem_stopwords = proc_binary.stopwords(lemmatized)

print("Lemmatized + stopwords: ", lem_stopwords[:3], '\n')

# Vectorization

vec_proccesses = {
    "X_binary_lemmatized" : proc_binary.vectorize(lemmatized),
    "X_bow_lemmatized"    : proc_bow.vectorize(lemmatized),
    "X_tf_idf_lemmatized" : proc_tf_idf.vectorize(lemmatized),
    
    "X_binary_stemmed" : proc_binary.vectorize(stemmed),
    "X_bow_stemmed"    : proc_bow.vectorize(stemmed),
    "X_tf_idf_stemmed" : proc_tf_idf.vectorize(stemmed),
    
    "X_binary_stopwords" : proc_binary.vectorize(lem_stopwords),
    "X_bow_stopwords"    : proc_bow.vectorize(lem_stopwords),
    "X_tf_idf_stopwords" : proc_tf_idf.vectorize(lem_stopwords)
}

print('saving processes...')
for name, X in vec_proccesses.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    print(f"{name} shape: {X.shape}")
    np.save(f"X_train_{name}.npy", X_train)
    np.save(f"X_test_{name}.npy", X_test)


# Train-test split (stratified)
print("Train distribution:", np.bincount(y_train))
print("Test distribution:", np.bincount(y_test))

np.save(f"y_train.npy", y_train)
np.save(f"y_test.npy", y_test)


Raw: ['How unhappy  some dogs like it though', "talking to my over driver about where I'm goinghe said he'd love to go to New York too but since Trump it's probably not", "Does anybody know if the Rand's likely to fall against the dollar? I got some money  I need to change into R but it keeps getting stronger unhappy "] 

Lemmatized: [['how', 'unhappy', 'some', 'dog', 'like', 'it', 'though'], ['talk', 'to', 'my', 'over', 'driver', 'about', 'where', 'i', 'be', 'goinghe', 'say', 'he', 'would', 'love', 'to', 'go', 'to', 'new', 'york', 'too', 'but', 'since', 'trump', 'it', 'be', 'probably', 'not'], ['do', 'anybody', 'know', 'if', 'the', 'rand', 'likely', 'to', 'fall', 'against', 'the', 'dollar', 'i', 'get', 'some', 'money', 'i', 'need', 'to', 'change', 'into', 'r', 'but', 'it', 'keep', 'get', 'strong', 'unhappy']] 

Stemmed:  [['how', 'unhappi', 'some', 'dog', 'like', 'it', 'though'], ['talk', 'to', 'my', 'over', 'driver', 'about', 'where', 'i', 'am', 'goingh', 'said', 'he', 'would', 'love